# LangChain: Building Applications with Language Models

## Introduction

**LangChain** is a powerful framework designed to simplify the development of applications powered by large language models (LLMs). It provides a comprehensive set of tools and abstractions that make it easier to:

- 🔗 **Chain together** different AI components and operations
- 🗄️ **Connect LLMs** to external data sources and APIs  
- 🧠 **Build memory** into conversational applications
- 🔧 **Create agents** that can use tools and make decisions
- 📊 **Process and retrieve** information from documents

### What You'll Learn

In this notebook, we'll explore:
1. **Memory Management** - How to efficiently handle GPU memory when working with models
2. **Local Model Integration** - Running quantized models locally using LangChain
3. **Model Parameters** - Understanding key configuration options
4. **Error Handling** - Common issues and how to resolve them

---

## Step 1: Memory Management for Apple Silicon

### Why Memory Management Matters

When working with large language models on Apple Silicon (M1/M2/M3) Macs, **Metal Performance Shaders (MPS)** provides GPU acceleration through PyTorch. However, GPU memory can become fragmented or not properly released between model loads, leading to:

- 🚫 **Out of memory errors** when loading new models
- 🐌 **Degraded performance** due to memory fragmentation  
- 💥 **Kernel crashes** in extreme cases

### The Solution: Cleanup Function

This utility function ensures that any previously loaded models and tokenizers are deleted and GPU memory is cleared. It's particularly important when:

- **Re-running notebook cells** that load models
- **Switching between different models** in the same session
- **Experimenting with different model configurations**

**Best Practice:** Always run this cleanup before loading new models to avoid memory-related issues.

In [5]:
def cleanup_mps_memory():
    """
    Frees MPS memory by deleting global variables 'model' and 'tokenizer' if they exist.
    Useful when you want to avoid passing model/tokenizer manually.
    """
    import gc
    import torch

    for var in ['model', 'tokenizer', 'pipe']:
        if var in globals():
            print(f"🔹 Deleting: {var}")
            del globals()[var]

    gc.collect()
    torch.mps.empty_cache()
    print("MPS memory cleaned.")
cleanup_mps_memory()

MPS memory cleaned.


## Step 2: Loading Quantized Models with LangChain

### What are Quantized Models?

**Quantization** is a technique that reduces the memory footprint and computational requirements of language models by using lower-precision numbers (e.g., 8-bit or 16-bit instead of 32-bit). This makes it possible to:

- 🏠 **Run large models locally** on consumer hardware
- ⚡ **Faster inference** due to reduced computational complexity
- 💾 **Lower memory usage** - models can be 2-4x smaller
- 🔋 **Better power efficiency** on laptops and mobile devices

### GGUF Format

**GGUF** (GPT-Generated Unified Format) is a popular format for quantized models that:
- Stores model weights in a compressed, efficient format
- Includes metadata about the model architecture
- Optimized for inference with libraries like `llama.cpp`

### LangChain + LlamaCpp Integration

LangChain provides a `LlamaCpp` wrapper that makes it easy to integrate GGUF models into your applications. The key parameters include:

- **`model_path`**: Path to your GGUF model file
- **`n_ctx`**: Context window size (how many tokens the model can process at once)
- **`max_tokens`**: Maximum number of tokens to generate in a response
- **`seed`**: For reproducible outputs (same input → same output)
- **`verbose`**: Whether to show detailed loading information

In [6]:
import os
model_path = os.path.expanduser("~/Downloads/Phi-3-mini-4k-instruct-q4.gguf")
model_path

'/Users/mayia/Downloads/Phi-3-mini-4k-instruct-q4.gguf'

In [7]:
# For GPU/Neural Engine acceleration on macOS, 
# make sure to install the latest version of llama-cpp-python from source 
# and enabling Metal support via CMake flag -DLLAMA_METAL=on: 
# CMAKE_ARGS="-DLLAMA_METAL=on" pip install llama-cpp-python --no-binary llama-cpp-python

from langchain import LlamaCpp  # import the LlamaCpp class from the langchain library

llm = LlamaCpp(
    model_path=model_path,  # path to the model file, model should be downloaded locally
    max_tokens=500,  # max tokens to generate
    n_ctx=2048,  # context window
    seed=42,  # random seed
    verbose=True   # verbose output, i.e. print the prompt, tokens, and output
)

llama_model_load_from_file_impl: using device Metal (Apple M4) - 16383 MiB free
llama_model_loader: loaded meta data with 24 key-value pairs and 195 tensors from /Users/mayia/Downloads/Phi-3-mini-4k-instruct-q4.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = phi3
llama_model_loader: - kv   1:                               general.name str              = Phi3
llama_model_loader: - kv   2:                        phi3.context_length u32              = 4096
llama_model_loader: - kv   3:                      phi3.embedding_length u32              = 3072
llama_model_loader: - kv   4:                   phi3.feed_forward_length u32              = 8192
llama_model_loader: - kv   5:                           phi3.block_count u32              = 32
llama_model_loader: - kv   6:                  phi3.attention.head_count u32   

In [8]:
from langchain import PromptTemplate
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(template=template, input_variables=["input_prompt"])
basic_chain = prompt | llm
basic_chain.invoke({"input_prompt": "What is quantum entanglement?"})

/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
llama_perf_context_print:        load time =     309.98 ms
llama_perf_context_print: prompt eval time =     309.75 ms /    12 tokens (   25.81 ms per token,    38.74 tokens per second)
llama_perf_context_print:        eval time =    9197.70 ms /   264 runs   (   34.84 ms per token,    28.70 tokens per second)
llama_perf_context_print:       total time =    9579.25 ms /   276 tokens


' Quantum entanglement is a physical phenomenon that occurs when pairs or groups of particles are generated, interact, or share spatial proximity in such a way that the quantum state of each particle cannot be described independently of the state of the others, even when the particles are separated by large distances. This interconnectedness means that the state of one entangled particle instantly influences the state of the other, no matter how far apart they are. Entanglement is a fundamental aspect of quantum mechanics and has been experimentally confirmed many times since its theoretical prediction in the 1930s by Albert Einstein, Boris Podolsky, and Nathan Rosen (EPR), followed by John Stewart Bell\'s theorem that challenged local hidden variable theories.\n\nEntangled particles are often referred to as "quantum links." One of the most famous examples of entanglement is the Einstein-Podolsky-Rosen paradox, also known as the EPR paradox, which questioned whether quantum mechanics w

## Step 3: Building Multi-Step Chains

### Understanding LangChain Chains

A **chain** in LangChain represents a sequence of operations that are executed in order, where the output of one step becomes the input to the next. This is one of LangChain's most powerful features for building complex AI workflows.

### Multi-Prompt Chains

In this example, we'll create a **story generation pipeline** with three connected steps:

1. **Title Generation** - Create a compelling title based on a story summary
2. **Character Development** - Develop a main character description using the title and summary  
3. **Story Writing** - Generate the complete story using all previous outputs

### Why Use Chains?

- 🔄 **Modularity**: Each step has a specific, focused purpose
- 🧩 **Reusability**: Individual prompts can be reused in different contexts
- 📊 **Debugging**: Easy to test and modify individual components
- 🎯 **Quality Control**: Each step can be optimized independently

### LLMChain vs Modern Approach

**Note**: The `LLMChain` class shown below is deprecated in favor of the newer `RunnableSequence` syntax (using `|` operator). We'll show both approaches for learning purposes.

### Step 3.1: Creating the Title Generation Chain

This first chain takes a story summary and generates an appropriate title. Key components:

- **Template**: Uses the Phi-3 model's specific chat format with `<|user|>` and `<|assistant|>` tokens
- **Input Variable**: `{summary}` - will be replaced with the actual story summary
- **Output Key**: `"title"` - allows us to reference this chain's output in later steps
- **Instruction**: "only return a title" - helps ensure clean, focused output


In [9]:
from langchain import LLMChain
template = """<s><|user|>
create a title for a story about a {summary}. only return a title. <|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

/var/folders/rs/c1gfqbv946s_1qry1fcqhzwc0000gn/T/ipykernel_91062/4112159.py:6: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


### Step 3.2: Creating the Character Development Chain

The second chain builds upon the first by using both the original summary AND the generated title:

- **Multiple Inputs**: `{summary}` and `{title}` - demonstrating how chains can use outputs from previous steps
- **Length Control**: "Use only two sentences" - constrains output for consistency
- **Output Key**: `"character"` - stores the character description for the final step


In [10]:
template = """<s><|user|>
Describe the main character of a story about a {summary} with a title {title}. 
Use only two sentences. <|end|>
<|assistant|>"""
character_prompt = PromptTemplate(template=template, input_variables=["summary", "title"])
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

### Step 3.3: Creating the Final Story Generation Chain

The final chain combines all previous outputs to create the complete story:

- **All Variables**: Uses `{summary}`, `{title}`, and `{character}` from previous steps
- **Output Constraint**: "Only return the story and it cannot be longer than 1 paragraph"
- **Complete Context**: The model now has rich context to generate a coherent story


In [11]:
template = """<s><|user|>
Create a story about {summary} with a title {title}. The main character is {character}.
Only return the story and it cannot be longer than 1 paragraph. <|end|>
<|assistant|>"""
story_prompt = PromptTemplate(template=template, input_variables=["summary", "title", "character"])
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

### Step 3.4: Chaining Everything Together

Here we see the power of LangChain's pipe operator (`|`):

```python
llm_chain = title | character | story
```

This creates a **sequential pipeline** where:
1. The input first goes to the `title` chain
2. The output (plus original input) flows to the `character` chain  
3. Finally, all accumulated outputs go to the `story` chain

### Execution Flow

When we call `invoke()` with our story summary, here's what happens:
1. **Title Generation**: "Bimka and Dymka..." → generates title
2. **Character Development**: Uses summary + title → creates character description
3. **Story Creation**: Uses summary + title + character → writes final story

**Note**: The performance output shows GPU utilization and token processing speeds for each step in the chain.


In [12]:
llm_chain = title | character | story
llm_chain.invoke("Bimka and Dymka are best cat friends. They are always together. They are both very cute and fluffy.")

/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Llama.generate: 3 prefix-match hit, remaining 44 prompt tokens to eval
llama_perf_context_print:        load time =     309.98 ms
llama_perf_context_print: prompt eval time =     808.88 ms /    44 tokens (   18.38 ms per token,    54.40 tokens per second)
llama_perf_context_print:        eval time =     706.49 ms /    22 runs   (   32.11 ms per token,    31.14 tokens per second)
llama_perf_context_print:       total time =    1519.58 ms /    66 tokens
/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Llama.generate: 3 prefix-match hit, remaining 74 prompt tokens to eval
llama_perf_context_print:      

{'summary': 'Bimka and Dymka are best cat friends. They are always together. They are both very cute and fluffy.',
 'title': ' "Furry Pals: Bimka and Dymka\'s Everlasting Cat Bond"',
 'character': ' Bimka and Dymka, two adorable and fluffy kittens with striking personalities, share an unbreakable bond as best friends. Together, they navigate the whimsical adventures of their feline world, showcasing the enduring power of friendship in "Furry Pals: Bimka and Dymka\'s Everlasting Cat Bond."',
 'story': ' Bimka and Dymka, two irresistibly cute and fluffy kittens with contrasting yet complementary personalities, share an unbreakable bond that transcends all boundaries. Their story unfolds in the heartwarming tale of "Furry Pals: Bimka and Dymka\'s Everlasting Cat Bond," where they embark on whimsical adventures through their feline world, demonstrating the enduring power of friendship. From playful frolicking under sunlit windowsills to daring explorations into hidden corners filled with m

## Step 4: Memory Management in Conversational AI

### The Problem: Stateless Language Models

By default, language models are **stateless** - they don't remember previous interactions. Each query is processed independently, which means:

- 🚫 **No conversation context** - can't refer to earlier messages
- 🔄 **No continuity** - each response starts from scratch  
- 💭 **No learning** - can't build on previous exchanges

### LangChain Memory Solutions

LangChain provides several memory mechanisms to make conversations feel natural and contextual:

1. **ConversationBufferMemory** - Stores all conversation history
2. **ConversationBufferWindowMemory** - Stores only recent messages (sliding window)
3. **ConversationSummaryMemory** - Maintains a summary of the conversation

### Conversation Buffer Memory

This is the simplest form of memory that stores the **complete conversation history**. Every message (both user inputs and AI responses) is preserved and included in subsequent prompts.

In [13]:
template = """<s><|user|>Current conversation:{chat_history}
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(template=template, input_variables=["chat_history", "input_prompt"])

### Step 4.1: Setting Up Conversation Buffer Memory

Key components of this memory setup:

- **Template Update**: Now includes `{chat_history}` variable to inject previous conversation
- **Memory Object**: `ConversationBufferMemory` with `memory_key="chat_history"`
- **Integration**: The `LLMChain` now includes the memory parameter

### How It Works

1. **First Message**: Chat history is empty, so only the new input is processed
2. **Subsequent Messages**: Previous exchanges are automatically included in the prompt
3. **Context Building**: Each response builds upon the full conversation history


In [14]:
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(memory_key="chat_history")
llm_chain = LLMChain(prompt=prompt, llm=llm, memory=memory)
llm_chain.invoke({"input_prompt": "Hi I am flying to New York, and I am hungry. What should I eat?"})

/var/folders/rs/c1gfqbv946s_1qry1fcqhzwc0000gn/T/ipykernel_91062/2024913130.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history")
/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Llama.generate: 3 prefix-match hit, remaining 25 prompt tokens to eval
llama_perf_context_print:        load time =     309.98 ms
llama_perf_context_print: prompt eval time =     561.43 ms /    25 tokens (   22.46 ms per token,    44.53 tokens per second)
llama_perf_context_print:        eval time =   16493.17 ms /   466 runs   (   35.39 ms per token,    28.25 tokens per second)
llama_perf_context_print:       total time =   17239.80 ms /   491 tokens


{'input_prompt': 'Hi I am flying to New York, and I am hungry. What should I eat?',
 'chat_history': '',
 'text': " Hey! When you're hungry, New York has an incredible food scene to explore. Here are a few suggestions:\n\n1. Grab some classic New York-style hot dogs from street vendors like Nathan's or Katz's for a quick bite at $5-$8. \n2. Visit one of the many food carts and trucks offering mouthwatering options, such as shrimp trucks (e.g., Fish Taco Shop), falafel stands, or dessert carts selling unique treats like cronuts or pistachio-topped pastries.\n3. Consider trying some iconic New York cheesecake at a local bakery for about $7-$10 per slice.\n4. For a sit-down meal, you could explore diverse cuisines in various neighborhoods:\n   - In Brooklyn (e.g., Bushwick or Williamsburg), try Ethiopian food, Middle Eastern dishes, or artisanal pizza at spots like Delfina or Di Fara Pizza. Prices range from $10-$25 per person.\n   - If you're in Lower Manhattan (e.g., Chinatown), indulge

### Step 4.2: Testing Memory Persistence

This second message demonstrates how the AI "remembers" the context:

- **Previous Context**: The AI recalls you're flying to New York and are hungry
- **New Information**: You mention being vegetarian
- **Contextual Response**: Recommendations are tailored to both location AND dietary preference

**Memory in Action**: The model can now provide restaurant suggestions specifically for vegetarian options in New York, showing how conversation memory enables more helpful, contextual responses.


In [15]:
# as we can see, our LLM still remebers that we are flying to New York
llm_chain.invoke({"input_prompt": "I am a vegetarian. What is a good restaurant there?"})

/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Llama.generate: 6 prefix-match hit, remaining 508 prompt tokens to eval
llama_perf_context_print:        load time =     309.98 ms
llama_perf_context_print: prompt eval time =   10752.72 ms /   508 tokens (   21.17 ms per token,    47.24 tokens per second)
llama_perf_context_print:        eval time =   14080.77 ms /   340 runs   (   41.41 ms per token,    24.15 tokens per second)
llama_perf_context_print:       total time =   24956.90 ms /   848 tokens


{'input_prompt': 'I am a vegetarian. What is a good restaurant there?',
 'chat_history': "Human: Hi I am flying to New York, and I am hungry. What should I eat?\nAI:  Hey! When you're hungry, New York has an incredible food scene to explore. Here are a few suggestions:\n\n1. Grab some classic New York-style hot dogs from street vendors like Nathan's or Katz's for a quick bite at $5-$8. \n2. Visit one of the many food carts and trucks offering mouthwatering options, such as shrimp trucks (e.g., Fish Taco Shop), falafel stands, or dessert carts selling unique treats like cronuts or pistachio-topped pastries.\n3. Consider trying some iconic New York cheesecake at a local bakery for about $7-$10 per slice.\n4. For a sit-down meal, you could explore diverse cuisines in various neighborhoods:\n   - In Brooklyn (e.g., Bushwick or Williamsburg), try Ethiopian food, Middle Eastern dishes, or artisanal pizza at spots like Delfina or Di Fara Pizza. Prices range from $10-$25 per person.\n   - If y

## Step 5: Windowed Conversation Memory

### The Challenge with Full Buffer Memory

While `ConversationBufferMemory` preserves all context, it has limitations:

- 📈 **Growing Context**: Conversations become longer and longer
- 💰 **Increased Costs**: More tokens = higher API costs (for cloud models)
- 🐌 **Slower Processing**: Longer prompts take more time to process
- 🚫 **Context Limits**: Models have maximum context window sizes

### Solution: Conversation Window Memory

`ConversationBufferWindowMemory` solves this by maintaining only the **most recent N exchanges**:

- **Parameter `k=2`**: Keeps only the last 2 conversation turns
- **Sliding Window**: Older messages automatically drop off
- **Balanced Approach**: Maintains recent context while controlling size

In [16]:
from langchain.memory import ConversationBufferWindowMemory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")
llm_chain = LLMChain(prompt=prompt, llm=llm, memory=memory)
llm_chain.predict(input_prompt="Hi I am Mikolaj and I am flying to New York, and I am hungry. What should I eat?")
llm_chain.predict(input_prompt="I am a vegetarian. What is a good restaurant there?")

/var/folders/rs/c1gfqbv946s_1qry1fcqhzwc0000gn/T/ipykernel_91062/4226982596.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")
/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Llama.generate: 6 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =     309.98 ms
llama_perf_context_print: prompt eval time =     494.43 ms /    28 tokens (   17.66 ms per token,    56.63 tokens per second)
llama_perf_context_print:        eval time =   14622.01 ms /   400 runs   (   36.56 ms per token,    27.36 tokens per second)
llama_perf_context_print:       total time =   15276.27 ms /   428 tokens
/opt/miniconda3/envs/re/lib/python3.10/sit

' New York City is a great place for vegetarians with its diverse food scene. Here are some top-rated vegetarian or vegan restaurants you can try:\n\n1. Blossom Restaurant - Offers a delicious plant-based menu focusing on creative dishes and flavors. Located in the East Village, their cuisine is inspired by different international flavors.\n\n2. Dirt Candy - A New York City institution that serves innovative vegetarian tasting menus with artistic presentations of vegan food. This restaurant can be found on the Lower East Side.\n\n3. Red Bamboo - Serves a variety of fresh and flavorful Asian-inspired dishes, including some excellent vegetarian options like tofu stir fry or cashew chicken (without the meat). Located in Chelsea, they are also open for takeout.\n\n4. Candle 79 - A popular upscranny vegan restaurant with a sophisticated atmosphere and inventive dishes made from organic ingredients. It is located on West 58th Street between Eighth Avenue and Broadway in the heart of Midtown 

### Step 5.1: Windowed Memory in Action

Let's trace through the conversation to see how windowed memory works:

1. **First Message**: "Hi I am Mikolaj..." - stored in memory
2. **Second Message**: "I am a vegetarian..." - now we have 2 exchanges (at limit)
3. **Third Message**: "Where do i fly?" - the FIRST exchange gets dropped

### Expected Behavior

- ✅ **Recent Context**: Still remembers vegetarian preference and New York destination
- ❌ **Forgotten Name**: The name "Mikolaj" should be forgotten (from the dropped first exchange)

This demonstrates the trade-off: we maintain efficiency while potentially losing older but important information.


In [17]:
llm_chain.invoke({"input_prompt":"Where do i fly?"})

/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Llama.generate: 439 prefix-match hit, remaining 528 prompt tokens to eval
llama_perf_context_print:        load time =     309.98 ms
llama_perf_context_print: prompt eval time =   11562.25 ms /   528 tokens (   21.90 ms per token,    45.67 tokens per second)
llama_perf_context_print:        eval time =   21481.84 ms /   450 runs   (   47.74 ms per token,    20.95 tokens per second)
llama_perf_context_print:       total time =   33242.18 ms /   978 tokens


{'input_prompt': 'Where do i fly?',
 'chat_history': 'Human: Hi I am Mikolaj and I am flying to New York, and I am hungry. What should I eat?\nAI:  Hello Mikolaj! Considering you\'re flying to New York, there are plenty of delicious options for both airport and city eateries. Here are a few suggestions:\n\n1. Airport food: If it\'s during your layover or before your flight, consider the following at JFK International Airport -\n   - Shake Shack for classic New York-style burgers and shakes.\n   - The No Parking Food Hall (in Terminal 4) offers a wide variety of options including street food favorites like El Chulo Taqueria or Sweetgreen salads.\n\n2. Once you\'re in New York:\n   - Try classic American fare such as cheeseburgers, hot dogs, and pizza slices from famous spots like Katz\'s Deli or Di Fara Pizza.\n   - For a more casual lunch option, grab a slice at Joe\'s Pizza in Brooklyn or a delicious bagel with cream cheese at Ess-a-Bagel on the Upper East Side.\n   - If you have an a

### Step 5.2: Testing the Memory Window

This query tests whether the model still has access to information from the first exchange:

**Question**: "Where do i fly?"
**Context**: This information was mentioned in the very first message

The model should still know about New York because that information was mentioned in multiple exchanges within the window.


In [18]:
llm_chain.invoke({"input_prompt":"What is my name?"})

/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Llama.generate: 9 prefix-match hit, remaining 988 prompt tokens to eval
llama_perf_context_print:        load time =     309.98 ms
llama_perf_context_print: prompt eval time =   19514.77 ms /   988 tokens (   19.75 ms per token,    50.63 tokens per second)
llama_perf_context_print:        eval time =    2599.90 ms /    55 runs   (   47.27 ms per token,    21.15 tokens per second)
llama_perf_context_print:       total time =   22129.35 ms /  1043 tokens


{'input_prompt': 'What is my name?',
 'chat_history': 'Human: I am a vegetarian. What is a good restaurant there?\nAI:  New York City is a great place for vegetarians with its diverse food scene. Here are some top-rated vegetarian or vegan restaurants you can try:\n\n1. Blossom Restaurant - Offers a delicious plant-based menu focusing on creative dishes and flavors. Located in the East Village, their cuisine is inspired by different international flavors.\n\n2. Dirt Candy - A New York City institution that serves innovative vegetarian tasting menus with artistic presentations of vegan food. This restaurant can be found on the Lower East Side.\n\n3. Red Bamboo - Serves a variety of fresh and flavorful Asian-inspired dishes, including some excellent vegetarian options like tofu stir fry or cashew chicken (without the meat). Located in Chelsea, they are also open for takeout.\n\n4. Candle 79 - A popular upscranny vegan restaurant with a sophisticated atmosphere and inventive dishes made f

### Step 5.3: Testing Information Loss

**Question**: "What is my name?"
**Expected Result**: The model should NOT know the name "Mikolaj"

This demonstrates the key limitation of windowed memory:
- **Information Loss**: Important details from early conversation are forgotten
- **Window Effect**: Only the most recent `k=2` exchanges are retained
- **Trade-off**: Memory efficiency vs. information completeness

**Use Cases for Windowed Memory**:
- Long-running conversations where only recent context matters
- Cost-sensitive applications 
- Real-time chat applications with memory constraints


## Step 6: Conversation Summary Memory

### The Best of Both Worlds

`ConversationSummaryMemory` provides an intelligent compromise between full buffer and windowed memory:

- 🧠 **Smart Compression**: Uses the LLM itself to summarize old conversations
- 💾 **Persistent Context**: Important information is retained in summary form
- ⚡ **Efficiency**: Maintains bounded context size
- 🎯 **Relevance**: Keeps essential details while discarding noise

### How It Works

1. **Recent Messages**: Last few exchanges stored verbatim (like windowed memory)
2. **Older Content**: Automatically summarized by the LLM
3. **Dynamic Summarization**: As conversation grows, older summaries get re-summarized
4. **Context Preservation**: Key information persists across the entire conversation

### Benefits

- **Memory Efficiency**: Constant context size regardless of conversation length
- **Information Retention**: Important details preserved in summary
- **Cost Control**: Predictable token usage for long conversations
- **Quality Maintenance**: LLM-generated summaries maintain coherence

**Note**: This approach requires additional LLM calls for summarization, which adds some computational overhead but provides superior long-term memory management.

In [19]:
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines. 
Current summary: {summary}
New lines of conversations: {new_lines}
New summary:<|end\>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["summary", "new_lines"],
    template=summary_prompt_template
)

In [20]:
from langchain.memory import ConversationSummaryMemory

memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt,
)

llm_chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
)

llm_chain.invoke({"input_prompt": "Hi I am Mikolaj and I am flying to New York, and I am hungry. What should I eat?"})
llm_chain.invoke({"input_prompt": "What's my name?"})

/var/folders/rs/c1gfqbv946s_1qry1fcqhzwc0000gn/T/ipykernel_91062/3569024165.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(
/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Llama.generate: 6 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =     309.98 ms
llama_perf_context_print: prompt eval time =     522.79 ms /    28 tokens (   18.67 ms per token,    53.56 tokens per second)
llama_perf_context_print:        eval time =   20328.85 ms /   499 runs   (   40.74 ms per token,    24.55 tokens per second)
llama_perf_context_print:       total time =   21116.39 ms /   527 tokens
/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: 

{'input_prompt': "What's my name?",
 'chat_history': " New summary:\nMikolaj, a traveler flying to New York and feeling hungry, seeks suggestions for meals. The AI recommends exploring various options such as street food (e.g., hot dogs, bagels), dining at diverse restaurants offering American, Italian, Chinese, and ethnic cuisines, visiting coffee shops/bakeries like Peet's Coffee & Tea or Sfoglia Bakery, or shopping from grocery stores for fresh ingredients. The AI highlights iconic New York establishments such as Katz's Deli and Lombardi's Pizza while also suggesting fast casual chains like Shake Shack and Chipotle Mexican Grill.",
 'text': ' Your name is Mikolaj.'}

In [22]:
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Llama.generate: 3 prefix-match hit, remaining 179 prompt tokens to eval
llama_perf_context_print:        load time =     309.98 ms
llama_perf_context_print: prompt eval time =    2878.77 ms /   179 tokens (   16.08 ms per token,    62.18 tokens per second)
llama_perf_context_print:        eval time =     570.04 ms /    17 runs   (   33.53 ms per token,    29.82 tokens per second)
llama_perf_context_print:       total time =    3452.25 ms /   196 tokens
/opt/miniconda3/envs/re/lib/python3.10/site-packages/llama_cpp/llama.py:1240: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
Llama.generate: 3 prefix-match hit, remaining 233 prompt tokens to eval
llama_perf_context_print:    

{'input_prompt': 'What was the first question I asked?',
 'chat_history': " New summary:\nMikolaj, a traveler flying to New York and feeling hungry, seeks meal suggestions. The AI recommends exploring street food options like hot dogs and bagels, diverse restaurants with American, Italian, Chinese, and ethnic cuisines, popular coffee shops/bakeries such as Peet's Coffee & Tea or Sfoglia Bakery, and grocery shopping for fresh ingredients. Iconic New York establishments like Katz's Deli and Lombardi's Pizza are also suggested, along with fast casual chains such as Shake Shack and Chipotle Mexican Grill. Additionally, the AI confirms Mikolaj's name when asked by the user.",
 'text': ' The first question you asked was: "Hi Mikolaj, how are you?"'}

In [24]:
memory.load_memory_variables({})

{'chat_history': ' New summary:\nMikolaj, traveling to New York and hungry, inquires about meal suggestions. The AI recommends exploring street food options like hot dogs and bagels, diverse restaurants with American, Italian, Chinese, and ethnic cuisines, popular coffee shops/bakeries such as Peet\'s Coffee & Tea or Sfoglia Bakery, and grocery shopping for fresh ingredients. Iconic New York establishments like Katz\'s Deli and Lombardi\'s Pizza are also suggested, along with fast casual chains such as Shake Shack and Chipotle Mexican Grill. The AI confirms Mikolaj\'s name upon request. Additionally, the first question asked by the user was "Hi Mikolaj, how are you?"'}

## Summary: LangChain Key Concepts

### What We've Learned

This notebook demonstrated several fundamental LangChain concepts for building robust AI applications:

#### 1. **Memory Management** 🧠
- **MPS Cleanup**: Essential for Apple Silicon users to avoid GPU memory issues
- **Best Practice**: Always clean memory before loading new models

#### 2. **Local Model Integration** 🏠
- **GGUF Format**: Efficient quantized models for local inference
- **LlamaCpp Integration**: Seamless LangChain wrapper for llama.cpp models
- **Performance Monitoring**: Understanding token processing speeds and GPU utilization

#### 3. **Chain Building** 🔗
- **Sequential Processing**: Using `|` operator to connect multiple steps
- **Output Propagation**: How results flow between chain components
- **Modular Design**: Building reusable, focused components

#### 4. **Memory Strategies** 💭
- **ConversationBufferMemory**: Complete history retention
- **ConversationBufferWindowMemory**: Recent context with size limits
- **ConversationSummaryMemory**: Intelligent compression with LLM summarization

### Choosing the Right Memory Strategy

| Memory Type | Best For | Pros | Cons |
|-------------|----------|------|------|
| **Buffer** | Short conversations | Complete context | Growing costs/latency |
| **Window** | Recent context focus | Predictable size | Information loss |
| **Summary** | Long conversations | Smart compression | Additional LLM calls |

### Next Steps

- **Experiment** with different memory configurations for your use case
- **Optimize** prompt templates for your specific model
- **Monitor** performance and memory usage in production
- **Explore** advanced features like agents and retrieval-augmented generation (RAG)

**Happy building with LangChain! 🚀**
